In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
import scanpy as sc
import anndata as ad
import cellrank as cr

print("Python   :", sys.version.split()[0])
print("CellRank :", cr.__version__)
print("Scanpy   :", sc.__version__)
print("AnnData  :", ad.__version__)
print("NumPy    :", np.__version__)
print("Pandas   :", pd.__version__)
print("SciPy    :", scipy.__version__)

In [ ]:
from pathlib import Path

INPUT_H5AD = Path(
    r"E:\TCGA\LIHC_scRNA_Step27_Trajectory\Data"
    r"\GSE149614_STEP27_FINAL_TRAJECTORY.h5ad"
)

adata28 = sc.read_h5ad(INPUT_H5AD)

print(adata28.shape)

In [ ]:
STATE_COL = "final_malignant_state_id"
PATIENT_COL = "patient"
PT_KEY = "MS6_anchored_pseudotime"
CONN_KEY = "neighbors_lineage_clean_connectivities"

print("State column:", STATE_COL in adata28.obs.columns)
print("Patient column:", PATIENT_COL in adata28.obs.columns)
print("Pseudotime:", PT_KEY in adata28.obs.columns)
print("Frozen graph:", CONN_KEY in adata28.obsp)

print(
    "Pseudotime range:",
    float(adata28.obs[PT_KEY].min()),
    float(adata28.obs[PT_KEY].max())
)

In [ ]:
velocity_layers = [
    x for x in ["spliced", "unspliced", "ambiguous"]
    if x in adata28.layers
]

print("Velocity layers:", velocity_layers)

RNA_VELOCITY_AVAILABLE = (
    "spliced" in adata28.layers
    and
    "unspliced" in adata28.layers
)

print("RNA velocity available:", RNA_VELOCITY_AVAILABLE)

In [ ]:
import inspect
from cellrank.kernels import PseudotimeKernel

print("PseudotimeKernel:")
print(inspect.signature(PseudotimeKernel))

print("\ncompute_transition_matrix:")
print(
    inspect.signature(
        PseudotimeKernel.compute_transition_matrix
    )
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step28 Cell 7
# Output directories
# =============================================================================

from pathlib import Path

STEP28_DIR = Path(
    _hcc_path('LIHC_scRNA_Step28_CellRank')
)

TABLE_DIR = STEP28_DIR / "Tables"
FIG_DIR = STEP28_DIR / "Figures"
DATA_DIR = STEP28_DIR / "Data"
AUDIT_DIR = STEP28_DIR / "Audit"
MANIFEST_DIR = STEP28_DIR / "Manifest"

for d in [
    TABLE_DIR,
    FIG_DIR,
    DATA_DIR,
    AUDIT_DIR,
    MANIFEST_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

print("Step28 directories ready.")

In [ ]:
# =============================================================================
# Step28 Cell 8
# Construct pseudotime kernel using the frozen Step27 graph
# =============================================================================

from cellrank.kernels import PseudotimeKernel
from scipy import sparse

pk = PseudotimeKernel(
    adata28,
    time_key=PT_KEY,
    backward=False,
    conn_key=CONN_KEY,
    check_connectivity=False
)

print(pk)
print("Kernel shape:", pk.shape)

In [ ]:
# =============================================================================
# Step28 Cell 9
# Verify exact reuse of the frozen connectivity graph
# =============================================================================

A_frozen = adata28.obsp[CONN_KEY].tocsr()
A_kernel = pk.connectivities.tocsr()

assert A_frozen.shape == A_kernel.shape

diff = A_frozen - A_kernel

if diff.nnz == 0:
    max_abs_diff = 0.0
else:
    max_abs_diff = float(
        np.max(np.abs(diff.data))
    )

print("Frozen graph shape:", A_frozen.shape)
print("Kernel graph shape:", A_kernel.shape)
print("Maximum absolute difference:", max_abs_diff)

assert max_abs_diff < 1e-12

print("PASS — exact frozen connectivity graph reused.")

In [ ]:
# =============================================================================
# Step28 Cell 10
# Primary pseudotime-directed transition matrix
# =============================================================================

pk.compute_transition_matrix(
    threshold_scheme="hard",
    frac_to_keep=0.30,
    check_irreducibility=False,
    n_jobs=1,
    show_progress_bar=False
)

T = pk.transition_matrix.tocsr()

print("Transition matrix shape:", T.shape)
print("Non-zero entries:", T.nnz)

In [ ]:
# =============================================================================
# Step28 Cell 11
# Transition-matrix mathematical QC
# =============================================================================

row_sums = np.asarray(
    T.sum(axis=1)
).ravel()

qc_transition = pd.Series({
    "n_cells": T.shape[0],
    "nnz": T.nnz,
    "row_sum_min": row_sums.min(),
    "row_sum_median": np.median(row_sums),
    "row_sum_max": row_sums.max(),
    "max_abs_row_sum_error": np.max(
        np.abs(row_sums - 1.0)
    )
})

display(
    qc_transition.to_frame("Value")
)

assert np.allclose(
    row_sums,
    1.0,
    atol=1e-8
)

print("PASS — transition matrix is row stochastic.")

In [ ]:
# =============================================================================
# Step28 Cell 12
# Global directional transition QC
# =============================================================================

pt = adata28.obs[
    PT_KEY
].to_numpy(dtype=float)

T_coo = T.tocoo()

src = T_coo.row
dst = T_coo.col
weight = T_coo.data

delta_pt = (
    pt[dst] - pt[src]
)

tol = 1e-12

self_mask = (
    src == dst
)

forward_mask = (
    (~self_mask) &
    (delta_pt > tol)
)

backward_mask = (
    (~self_mask) &
    (delta_pt < -tol)
)

flat_mask = (
    (~self_mask) &
    (np.abs(delta_pt) <= tol)
)

mass = {
    "Self": weight[self_mask].sum(),
    "Forward": weight[forward_mask].sum(),
    "Backward": weight[backward_mask].sum(),
    "Equal_PT": weight[flat_mask].sum()
}

direction_qc = pd.DataFrame({
    "Transition_class": mass.keys(),
    "Total_mass": mass.values()
})

direction_qc[
    "Fraction_all"
] = (
    direction_qc["Total_mass"] /
    direction_qc["Total_mass"].sum()
)

display(direction_qc)

In [ ]:
# =============================================================================
# Step28 Cell 13
# Off-diagonal directional transition fractions
# =============================================================================

offdiag_total = (
    weight[~self_mask].sum()
)

forward_fraction = (
    weight[forward_mask].sum()
    / offdiag_total
)

backward_fraction = (
    weight[backward_mask].sum()
    / offdiag_total
)

equal_fraction = (
    weight[flat_mask].sum()
    / offdiag_total
)

direction_offdiag = pd.Series({
    "Forward_fraction": forward_fraction,
    "Backward_fraction": backward_fraction,
    "Equal_PT_fraction": equal_fraction,
    "Forward_minus_backward":
        forward_fraction - backward_fraction
})

display(
    direction_offdiag.to_frame("Value")
)

In [ ]:
# =============================================================================
# Step28 Cell 14
# State-to-state mean one-step transition probabilities
# =============================================================================

state_series = (
    adata28.obs[STATE_COL]
    .astype(str)
)

states = [
    f"MS{i}" for i in range(1, 9)
]

state_flow = pd.DataFrame(
    0.0,
    index=states,
    columns=states
)

for source_state in states:

    source_idx = np.where(
        state_series.values == source_state
    )[0]

    for target_state in states:

        target_idx = np.where(
            state_series.values == target_state
        )[0]

        state_flow.loc[
            source_state,
            target_state
        ] = float(
            T[source_idx][
                :, target_idx
            ].sum()
            / len(source_idx)
        )

display(
    state_flow.round(4)
)

In [ ]:
# =============================================================================
# Step28 Cell 15
# State-flow QC
# =============================================================================

state_flow_row_sums = (
    state_flow.sum(axis=1)
)

display(
    state_flow_row_sums
    .to_frame("Row_sum")
)

assert np.allclose(
    state_flow_row_sums.values,
    1.0,
    atol=1e-6
)

print(
    "PASS — state-level transition matrix is normalized."
)

In [ ]:
# =============================================================================
# Step28 Cell 16
# Reciprocal transition flow along the frozen MS6-MS5-MS3 axis
# =============================================================================

axis_pairs = [
    ("MS6", "MS5"),
    ("MS5", "MS3"),
    ("MS6", "MS3")
]

axis_flow_rows = []

for source, target in axis_pairs:

    forward = state_flow.loc[
        source, target
    ]

    reverse = state_flow.loc[
        target, source
    ]

    axis_flow_rows.append({
        "Forward": f"{source}->{target}",
        "Forward_flow": forward,
        "Reverse": f"{target}->{source}",
        "Reverse_flow": reverse,
        "Net_forward":
            forward - reverse,
        "Forward_reverse_ratio":
            (forward + 1e-12)
            / (reverse + 1e-12)
    })

axis_flow = pd.DataFrame(
    axis_flow_rows
)

display(
    axis_flow.round(4)
)

In [ ]:
# =============================================================================
# Step28 Cell 17
# Store primary CellRank transition matrix
# =============================================================================

T_KEY = (
    "T_step28_pseudotime_primary"
)

assert T_KEY not in adata28.obsp

pk.write_to_adata(
    key=T_KEY
)

print(
    "Stored:",
    T_KEY
)

print(
    "Shape:",
    adata28.obsp[T_KEY].shape
)

In [ ]:
# =============================================================================
# Step28 Cell 18
# Save Step28B QC tables
# =============================================================================

direction_qc.to_csv(
    TABLE_DIR /
    "Step28B_Global_Directional_Mass.csv",
    index=False
)

direction_offdiag.to_frame(
    "Value"
).to_csv(
    TABLE_DIR /
    "Step28B_OffDiagonal_Direction_QC.csv"
)

state_flow.to_csv(
    TABLE_DIR /
    "Step28B_State_Transition_Matrix.csv"
)

axis_flow.to_csv(
    TABLE_DIR /
    "Step28B_MS6_MS5_MS3_Flow.csv",
    index=False
)

print(
    "Step28B tables saved."
)

In [ ]:
# =============================================================================
# Step28 Cell 19
# REPORT — Step28B primary transition matrix
# =============================================================================

print("=" * 100)
print("STEP28B PRIMARY PSEUDOTIME-KERNEL REPORT")
print("=" * 100)

print(
    "\nFrozen graph max difference:",
    max_abs_diff
)

print(
    "\nTransition matrix:"
)

display(
    qc_transition.to_frame("Value")
)

print(
    "\nOff-diagonal directional fractions:"
)

display(
    direction_offdiag.to_frame("Value")
)

print(
    "\nMS6-MS5-MS3 reciprocal flow:"
)

display(
    axis_flow.round(4)
)

print(
    "\nState transition matrix:"
)

display(
    state_flow.round(4)
)

print(
    "\nInterpretation boundary:"
)

print(
    "Direction is conditioned on the frozen "
    "Step27 pseudotime and is NOT independent "
    "temporal or lineage evidence."
)

print(
    "\nSTEP28B COMPLETE"
)

In [ ]:
# =============================================================================
# Step28 Cell 20
# Helper functions for transition robustness
# =============================================================================

def directional_metrics(T, pt, tol=1e-12):

    coo = T.tocoo()

    src = coo.row
    dst = coo.col
    w = coo.data

    delta = pt[dst] - pt[src]

    off = src != dst

    forward = off & (delta > tol)
    backward = off & (delta < -tol)
    equal = off & (np.abs(delta) <= tol)

    total = w[off].sum()

    return {
        "Forward_fraction":
            float(w[forward].sum() / total),

        "Backward_fraction":
            float(w[backward].sum() / total),

        "Equal_fraction":
            float(w[equal].sum() / total)
    }


def state_pair_flow(
    T,
    state_values,
    source,
    target
):

    source_idx = np.flatnonzero(
        state_values == source
    )

    target_idx = np.flatnonzero(
        state_values == target
    )

    return float(
        T[source_idx][:, target_idx].sum()
        / len(source_idx)
    )

In [ ]:
# =============================================================================
# Step28 Cell 21
# Pre-defined kernel sensitivity settings
# =============================================================================

kernel_settings = [
    {
        "Label": "Hard_f0.10",
        "scheme": "hard",
        "kwargs": {
            "frac_to_keep": 0.10
        }
    },
    {
        "Label": "Hard_f0.30_PRIMARY",
        "scheme": "hard",
        "kwargs": {
            "frac_to_keep": 0.30
        }
    },
    {
        "Label": "Hard_f0.50",
        "scheme": "hard",
        "kwargs": {
            "frac_to_keep": 0.50
        }
    },
    {
        "Label": "Soft_default",
        "scheme": "soft",
        "kwargs": {
            "b": 10.0,
            "nu": 0.5
        }
    }
]

pd.DataFrame(kernel_settings)

In [ ]:
# =============================================================================
# Step28 Cell 22
# Kernel parameter sensitivity
# =============================================================================

state_values = (
    adata28.obs[STATE_COL]
    .astype(str)
    .to_numpy()
)

pt_primary = (
    adata28.obs[PT_KEY]
    .to_numpy(dtype=float)
)

sensitivity_rows = []

for setting in kernel_settings:

    print(
        "Running:",
        setting["Label"]
    )

    k = PseudotimeKernel(
        adata28,
        time_key=PT_KEY,
        backward=False,
        conn_key=CONN_KEY,
        check_connectivity=False
    )

    k.compute_transition_matrix(
        threshold_scheme=
            setting["scheme"],
        n_jobs=1,
        show_progress_bar=False,
        check_irreducibility=False,
        **setting["kwargs"]
    )

    Tk = (
        k.transition_matrix
        .tocsr()
    )

    dm = directional_metrics(
        Tk,
        pt_primary
    )

    row = {
        "Setting":
            setting["Label"],

        **dm
    }

    for source, target in [
        ("MS6", "MS5"),
        ("MS5", "MS3"),
        ("MS6", "MS3")
    ]:

        fwd = state_pair_flow(
            Tk,
            state_values,
            source,
            target
        )

        rev = state_pair_flow(
            Tk,
            state_values,
            target,
            source
        )

        pair = (
            f"{source}_{target}"
        )

        row[
            f"{pair}_forward"
        ] = fwd

        row[
            f"{pair}_reverse"
        ] = rev

        row[
            f"{pair}_net"
        ] = (
            fwd - rev
        )

        row[
            f"{pair}_ratio"
        ] = (
            (fwd + 1e-12)
            /
            (rev + 1e-12)
        )

    sensitivity_rows.append(
        row
    )

kernel_sensitivity = pd.DataFrame(
    sensitivity_rows
)

display(
    kernel_sensitivity.round(4)
)

In [ ]:
# =============================================================================
# Step28 Cell 23
# Summarize directional robustness
# =============================================================================

robustness_summary = pd.DataFrame({
    "Criterion": [
        "Global forward fraction > 0.5",
        "MS6->MS5 net flow > 0",
        "MS5->MS3 net flow > 0",
        "MS6->MS3 net flow > 0"
    ],

    "N_support": [
        int(
            (
                kernel_sensitivity[
                    "Forward_fraction"
                ] > 0.5
            ).sum()
        ),

        int(
            (
                kernel_sensitivity[
                    "MS6_MS5_net"
                ] > 0
            ).sum()
        ),

        int(
            (
                kernel_sensitivity[
                    "MS5_MS3_net"
                ] > 0
            ).sum()
        ),

        int(
            (
                kernel_sensitivity[
                    "MS6_MS3_net"
                ] > 0
            ).sum()
        )
    ]
})

robustness_summary[
    "N_total"
] = len(kernel_sensitivity)

robustness_summary[
    "Support_fraction"
] = (
    robustness_summary["N_support"]
    /
    robustness_summary["N_total"]
)

display(
    robustness_summary
)

In [ ]:
# =============================================================================
# Step28 Cell 24
# Root-specific pseudotime keys
# =============================================================================

root_pt_keys = {
    "Global_MS6":
        "MS6_anchored_pseudotime",

    "HCC09_MS6":
        "dpt_pseudotime_HCC09MS6",

    "HCC10_MS6":
        "dpt_pseudotime_HCC10MS6"
}

for label, key in root_pt_keys.items():

    print(
        label,
        key,
        key in adata28.obs.columns
    )

In [ ]:
# =============================================================================
# Step28 Cell 25
# Root sensitivity of transition structure
# =============================================================================

root_rows = []

for root_label, time_key in root_pt_keys.items():

    print(
        "Running:",
        root_label
    )

    kr = PseudotimeKernel(
        adata28,
        time_key=time_key,
        backward=False,
        conn_key=CONN_KEY,
        check_connectivity=False
    )

    kr.compute_transition_matrix(
        threshold_scheme="hard",
        frac_to_keep=0.30,
        n_jobs=1,
        show_progress_bar=False,
        check_irreducibility=False
    )

    Tr = (
        kr.transition_matrix
        .tocsr()
    )

    ptr = (
        adata28.obs[time_key]
        .to_numpy(dtype=float)
    )

    dm = directional_metrics(
        Tr,
        ptr
    )

    row = {
        "Root":
            root_label,

        **dm
    }

    for source, target in [
        ("MS6", "MS5"),
        ("MS5", "MS3"),
        ("MS6", "MS3")
    ]:

        fwd = state_pair_flow(
            Tr,
            state_values,
            source,
            target
        )

        rev = state_pair_flow(
            Tr,
            state_values,
            target,
            source
        )

        pair = (
            f"{source}_{target}"
        )

        row[
            f"{pair}_forward"
        ] = fwd

        row[
            f"{pair}_reverse"
        ] = rev

        row[
            f"{pair}_net"
        ] = (
            fwd - rev
        )

    root_sensitivity = None

    root_rows.append(
        row
    )

root_sensitivity = pd.DataFrame(
    root_rows
)

display(
    root_sensitivity.round(4)
)

In [ ]:
# =============================================================================
# Step28 Cell 26
# Within-patient transition helper
# =============================================================================

patient_values = (
    adata28.obs[PATIENT_COL]
    .astype(str)
    .to_numpy()
)


def within_patient_flow(
    T,
    patient,
    source,
    target
):

    patient_mask = (
        patient_values == patient
    )

    source_mask = (
        patient_mask &
        (state_values == source)
    )

    target_mask = (
        patient_mask &
        (state_values == target)
    )

    source_idx = np.flatnonzero(
        source_mask
    )

    target_idx = np.flatnonzero(
        target_mask
    )

    patient_idx = np.flatnonzero(
        patient_mask
    )

    n_source = len(source_idx)

    same_patient_mass = float(
        T[source_idx][:, patient_idx]
        .sum()
        / n_source
    )

    target_mass = float(
        T[source_idx][:, target_idx]
        .sum()
        / n_source
    )

    conditional_flow = (
        target_mass /
        same_patient_mass
        if same_patient_mass > 0
        else np.nan
    )

    return (
        n_source,
        same_patient_mass,
        target_mass,
        conditional_flow
    )

In [ ]:
# =============================================================================
# Step28 Cell 27
# HCC09/HCC10 within-patient flow
# =============================================================================

patient_rows = []

for patient in [
    "HCC09",
    "HCC10"
]:

    for source, target in [
        ("MS6", "MS5"),
        ("MS5", "MS3"),
        ("MS6", "MS3")
    ]:

        (
            n_f,
            same_f,
            mass_f,
            cond_f
        ) = within_patient_flow(
            T,
            patient,
            source,
            target
        )

        (
            n_r,
            same_r,
            mass_r,
            cond_r
        ) = within_patient_flow(
            T,
            patient,
            target,
            source
        )

        patient_rows.append({
            "Patient":
                patient,

            "Pair":
                f"{source}->{target}",

            "N_forward_source":
                n_f,

            "Forward_same_patient_mass":
                same_f,

            "Forward_conditional_flow":
                cond_f,

            "Reverse_pair":
                f"{target}->{source}",

            "N_reverse_source":
                n_r,

            "Reverse_same_patient_mass":
                same_r,

            "Reverse_conditional_flow":
                cond_r,

            "Net_conditional_flow":
                cond_f - cond_r,

            "Forward_reverse_ratio":
                (
                    (cond_f + 1e-12)
                    /
                    (cond_r + 1e-12)
                )
        })

patient_flow = pd.DataFrame(
    patient_rows
)

display(
    patient_flow.round(4)
)

In [ ]:
# =============================================================================
# Step28 Cell 28
# Save Step28C tables
# =============================================================================

kernel_sensitivity.to_csv(
    TABLE_DIR /
    "Step28C_Kernel_Parameter_Sensitivity.csv",
    index=False
)

robustness_summary.to_csv(
    TABLE_DIR /
    "Step28C_Kernel_Robustness_Summary.csv",
    index=False
)

root_sensitivity.to_csv(
    TABLE_DIR /
    "Step28C_Root_Sensitivity.csv",
    index=False
)

patient_flow.to_csv(
    TABLE_DIR /
    "Step28C_Patient_Aware_Flow.csv",
    index=False
)

print(
    "Step28C tables saved."
)

In [ ]:
# =============================================================================
# Step28 Cell 29
# REPORT — Step28C robustness
# =============================================================================

print("=" * 100)
print("STEP28C TRANSITION ROBUSTNESS REPORT")
print("=" * 100)

print(
    "\n[1] Kernel parameter sensitivity"
)

display(
    kernel_sensitivity.round(4)
)

print(
    "\n[2] Robustness summary"
)

display(
    robustness_summary.round(4)
)

print(
    "\n[3] Root sensitivity"
)

display(
    root_sensitivity.round(4)
)

print(
    "\n[4] HCC09/HCC10 patient-aware flow"
)

display(
    patient_flow.round(4)
)

print(
    "\nInterpretation boundary:"
)

print(
    "All CellRank directions remain conditioned "
    "on Step27 pseudotime."
)

print(
    "Parameter and patient analyses test robustness, "
    "not independent temporal causality."
)

print(
    "\nSTEP28C COMPLETE"
)

In [ ]:
# =============================================================================
# Step28 Cell 30
# Initialize GPCCA from the PRIMARY pseudotime kernel
# =============================================================================

from cellrank.estimators import GPCCA

g = GPCCA(pk)

print(g)
print("Estimator shape:", g.shape)

In [ ]:
# =============================================================================
# Step28 Cell 31
# Sparse Schur decomposition
# =============================================================================

g.compute_schur(
    n_components=12,
    method="krylov",
    which="LR"
)

print("Schur decomposition complete.")
print("Schur vectors:", g.schur_vectors.shape)